# Notebook 04 — Entrenamiento y evaluación

---

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)

paquete = joblib.load(os.path.join(RUTA, "datos_preparados.joblib"))
X_train, X_test = paquete["X_train"], paquete["X_test"]
y_train, y_test = paquete["y_train"], paquete["y_test"]
cols_num, cols_cat = paquete["cols_num"], paquete["cols_cat"]
etiquetas = sorted(y_train.unique())

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Clases:", etiquetas)

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cols_cat),
])

## 3.1 Resultados del train (10 %)

Se entrenan y comparan las dos estrategias que la rúbrica plantea:

- **Softmax (multinomial).** Un único modelo que estima las tres probabilidades a la vez y las
  normaliza para que sumen 1. Es el comportamiento por defecto de `LogisticRegression` con el solver
  `lbfgs` cuando hay más de dos clases.
- **One-vs-Rest.** Tres modelos binarios independientes, uno por clase, cada uno respondiendo
  "¿es de mi clase o no?". Gana la clase cuyo modelo responde con mayor probabilidad.

Además se explora **`C`**, el inverso de la fuerza de regularización: valores chicos regularizan
mucho (modelo más simple), valores grandes regularizan poco (más flexible, más riesgo de sobreajuste).

In [ ]:
def construir_softmax(C, balanced=True):
    return Pipeline([
        ("prep", preprocesador),
        ("clf", LogisticRegression(
            C=C, solver="lbfgs", max_iter=2000,
            class_weight="balanced" if balanced else None,
            random_state=RANDOM_STATE)),
    ])

def construir_ovr(C, balanced=True):
    base = LogisticRegression(
        C=C, solver="lbfgs", max_iter=2000,
        class_weight="balanced" if balanced else None,
        random_state=RANDOM_STATE)
    return Pipeline([("prep", preprocesador), ("clf", OneVsRestClassifier(base))])

print("Cada ajuste tarda entre 10 y 30 segundos. El notebook completo, entre 3 y 6 minutos.")

### Verificación empírica de la estrategia de desbalance

Antes del barrido se contrasta el mismo modelo con y sin `class_weight="balanced"`.
Esta comparación es la evidencia que sostiene lo argumentado en la sección 2.4 del notebook 03.

In [ ]:
comparacion_peso = []
for balanced in [False, True]:
    m = construir_softmax(1.0, balanced=balanced)
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    comparacion_peso.append({
        "class_weight": "balanced" if balanced else "ninguno",
        "accuracy": accuracy_score(y_test, p),
        "f1_macro": f1_score(y_test, p, average="macro"),
        "recall_NoShow": recall_score(y_test, p, labels=["No-Show"], average="macro", zero_division=0),
        "precision_NoShow": precision_score(y_test, p, labels=["No-Show"], average="macro", zero_division=0),
        "veces_que_predijo_NoShow": int((p == "No-Show").sum()),
    })

tabla_peso = pd.DataFrame(comparacion_peso)
tabla_peso.round(4)

**Lectura de esta tabla — es uno de los hallazgos centrales del trabajo.**

Sin ponderación, el modelo **nunca predice `No-Show`**: su recall en esa clase es exactamente 0.
Obtiene una accuracy alta justamente por ignorarla, que es el comportamiento anticipado en el
análisis de desbalance. Con `class_weight="balanced"` el modelo empieza a señalar No-Shows y su
recall sube de forma marcada, pero la precisión en esa clase queda muy baja: emite muchas más
alertas de las que corresponden.

El F1 macro apenas se mueve entre ambas configuraciones, pero **el comportamiento del modelo cambia
por completo**. La elección no la resuelve la métrica sino el costo del negocio: si perder una noche
por un no-show cuesta más que molestar a varios clientes pidiéndoles garantía, conviene la versión
ponderada. Se adopta `class_weight="balanced"` por ese motivo, dejando constancia del intercambio.

### Barrido de hiperparámetros

Se prueban cinco valores de `C` sobre las dos estrategias: diez modelos en total.

In [ ]:
valores_C = [0.001, 0.01, 0.1, 1, 10]
registros = []

for C in valores_C:
    for nombre, constructor in [("Softmax", construir_softmax), ("One-vs-Rest", construir_ovr)]:
        modelo = constructor(C)
        modelo.fit(X_train, y_train)
        pred_tr = modelo.predict(X_train)
        pred_te = modelo.predict(X_test)
        registros.append({
            "estrategia": nombre, "C": C,
            "f1_macro_train": f1_score(y_train, pred_tr, average="macro"),
            "f1_macro_test": f1_score(y_test, pred_te, average="macro"),
            "accuracy_train": accuracy_score(y_train, pred_tr),
            "accuracy_test": accuracy_score(y_test, pred_te),
        })
        print(f"  {nombre:12s} C={C:<7} F1 macro test = {registros[-1]['f1_macro_test']:.4f}")

resultados = pd.DataFrame(registros)
resultados["brecha"] = resultados["f1_macro_train"] - resultados["f1_macro_test"]
resultados.round(4)

In [ ]:
mejor = resultados.loc[resultados["f1_macro_test"].idxmax()]
print("=== Mejor configuracion ===")
print(mejor.to_string(), "\n")

constructor = construir_softmax if mejor["estrategia"] == "Softmax" else construir_ovr
modelo_final = constructor(mejor["C"])
modelo_final.fit(X_train, y_train)

pred_train = modelo_final.predict(X_train)
print("=== Reporte de clasificacion — TRAIN ===")
print(classification_report(y_train, pred_train, labels=etiquetas, digits=3))

In [ ]:
cm_train = confusion_matrix(y_train, pred_train, labels=etiquetas)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_train, annot=True, fmt="d", cmap="Blues",
            xticklabels=etiquetas, yticklabels=etiquetas, ax=ax)
ax.set_title("Matriz de confusion — TRAIN")
ax.set_xlabel("Prediccion"); ax.set_ylabel("Valor real")
plt.tight_layout()
guardar("04_matriz_confusion_train")
plt.show()

## 3.2 Resultados del test (10 %)

El conjunto de prueba no intervino en ninguna etapa: ni en el ajuste del escalador, ni en la
codificación de categorías, ni en la selección de los países más frecuentes, ni en el entrenamiento.

In [ ]:
pred_test = modelo_final.predict(X_test)

print("=== Reporte de clasificacion — TEST ===")
print(classification_report(y_test, pred_test, labels=etiquetas, digits=3))

cm_test = confusion_matrix(y_test, pred_test, labels=etiquetas)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Oranges",
            xticklabels=etiquetas, yticklabels=etiquetas, ax=ax)
ax.set_title("Matriz de confusion — TEST")
ax.set_xlabel("Prediccion"); ax.set_ylabel("Valor real")
plt.tight_layout()
guardar("04_matriz_confusion_test")
plt.show()

In [ ]:
def fila_metricas(y_real, y_pred, etiqueta):
    return {
        "conjunto": etiqueta,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision_macro": precision_score(y_real, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_real, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_real, y_pred, average="macro"),
    }

comparativa = pd.DataFrame([
    fila_metricas(y_train, pred_train, "Train"),
    fila_metricas(y_test, pred_test, "Test"),
]).set_index("conjunto")
comparativa.loc["Diferencia"] = comparativa.loc["Train"] - comparativa.loc["Test"]
comparativa.round(4)

## 3.3 Prueba con features propios (10 %)

Se construyen dos reservas inventadas, con valores elegidos para representar perfiles opuestos según
lo observado en el notebook 02, y se evalúa si la predicción es coherente con el criterio del negocio.

- **Reserva A — perfil de bajo riesgo.** Poca anticipación, huésped recurrente, sin cancelaciones
  previas, con estacionamiento y pedidos especiales.
- **Reserva B — perfil de alto riesgo.** Reservada con casi un año de anticipación, sin depósito,
  con cancelaciones previas, sin ningún servicio adicional solicitado.

In [ ]:
plantilla = X_train.iloc[0].copy()

def crear_reserva(cambios):
    r = plantilla.copy()
    for k, v in cambios.items():
        r[k] = v
    return r

reserva_A = crear_reserva({
    "hotel": "City Hotel", "lead_time": 5,
    "arrival_date_year": 2017, "arrival_date_month": "March",
    "stays_in_weekend_nights": 1, "stays_in_week_nights": 2,
    "adults": 2, "children": 0, "babies": 0,
    "meal": "BB", "country": "PRT", "market_segment": "Direct",
    "distribution_channel": "Direct", "is_repeated_guest": 1,
    "previous_cancellations": 0, "previous_bookings_not_canceled": 3,
    "reserved_room_type": "A", "booking_changes": 0,
    "deposit_type": "No Deposit", "days_in_waiting_list": 0,
    "customer_type": "Transient", "adr": 95.0,
    "required_car_parking_spaces": 1, "total_of_special_requests": 2,
    "con_agente": 0,
})

reserva_B = crear_reserva({
    "hotel": "Resort Hotel", "lead_time": 340,
    "arrival_date_year": 2017, "arrival_date_month": "August",
    "stays_in_weekend_nights": 2, "stays_in_week_nights": 5,
    "adults": 2, "children": 0, "babies": 0,
    "meal": "BB", "country": "Otros", "market_segment": "Online TA",
    "distribution_channel": "TA/TO", "is_repeated_guest": 0,
    "previous_cancellations": 2, "previous_bookings_not_canceled": 0,
    "reserved_room_type": "A", "booking_changes": 0,
    "deposit_type": "No Deposit", "days_in_waiting_list": 0,
    "customer_type": "Transient", "adr": 180.0,
    "required_car_parking_spaces": 0, "total_of_special_requests": 0,
    "con_agente": 1,
})

mis_reservas = pd.DataFrame([reserva_A, reserva_B], index=["Reserva A", "Reserva B"])
mis_reservas[["hotel", "lead_time", "market_segment", "previous_cancellations",
              "required_car_parking_spaces", "total_of_special_requests", "adr"]]

In [ ]:
predicciones = modelo_final.predict(mis_reservas)
probabilidades = modelo_final.predict_proba(mis_reservas)

salida = pd.DataFrame(probabilidades, columns=modelo_final.classes_, index=mis_reservas.index)
salida["PREDICCION"] = predicciones
salida.round(3)

**Interpretación y coherencia con el dominio.**

Redactá esta parte después de ejecutar, contrastando con lo esperado:

- La Reserva A debería inclinarse hacia `Check-Out`. Reservar con cinco días de anticipación, ser
  huésped recurrente y pedir estacionamiento son señales de un viaje ya organizado.
- La Reserva B debería mostrar probabilidad bastante mayor de `Canceled`. Casi un año de
  anticipación, sin depósito y con cancelaciones previas es el perfil más volátil.
- Observá que `No-Show` casi nunca gana, aun con `class_weight="balanced"`. Es consistente con lo
  discutido: es una clase rara y poco separable de `Canceled`.

**Lectura de negocio:** la Reserva B es exactamente el caso en el que el hotel debería exigir
depósito o garantía de tarjeta.

### Guardado de resultados para el notebook 05

In [ ]:
joblib.dump({
    "modelo_final": modelo_final,
    "resultados": resultados,
    "tabla_peso": tabla_peso,
    "pred_train": pred_train,
    "pred_test": pred_test,
    "etiquetas": etiquetas,
    "mejor": mejor.to_dict(),
}, os.path.join(RUTA, "modelo_y_resultados.joblib"))

resultados.to_csv(os.path.join(RUTA, "barrido_hiperparametros.csv"), index=False)
print("Guardado en:", RUTA)

### Exportación del modelo para publicación

El archivo anterior pesa varios megabytes porque incluye los vectores de predicción sobre las 119.209
observaciones, útiles para el notebook 05 pero irrelevantes para distribuir el modelo. La celda
siguiente arma un paquete mínimo en `release/` con tres artefactos que cumplen propósitos distintos:

**`modelo_reservas_hotel.joblib`** contiene únicamente el `Pipeline` ajustado, es decir el
estandarizador, el codificador y los tres clasificadores binarios. Se carga con `joblib.load` y queda
listo para predecir. Su limitación es que un objeto serializado de scikit-learn depende de la versión
de la biblioteca con la que se creó, de modo que puede dejar de cargarse en versiones futuras.

**`modelo_web.json`** resuelve esa fragilidad. Guarda los parámetros aprendidos —medias y
desviaciones del estandarizador, categorías del codificador, coeficientes e interceptos— en un
formato de texto plano que no depende de scikit-learn ni de Python. Con él, la predicción se
reconstruye en cualquier lenguaje aplicando estandarización, codificación *one-hot*, la sigmoide de
cada clasificador binario y la normalización final que las convierte en probabilidades.

**`metricas.json`** registra la configuración seleccionada, el desempeño alcanzado, el tamaño de las
particiones y la versión de scikit-learn empleada, de modo que los resultados publicados sean
verificables sin necesidad de reejecutar el experimento.

La última parte de la celda **verifica que el JSON reproduzca exactamente las probabilidades de
scikit-learn** sobre doscientas observaciones del conjunto de prueba. Sin esa comprobación, el
artefacto portable sería una afirmación no verificada.

In [ ]:
import json, sklearn

CARPETA_RELEASE = os.path.join(RUTA, "release")
os.makedirs(CARPETA_RELEASE, exist_ok=True)

joblib.dump(modelo_final, os.path.join(CARPETA_RELEASE, "modelo_reservas_hotel.joblib"), compress=3)

ct = modelo_final.named_steps["prep"]
clf = modelo_final.named_steps["clf"]
scaler = ct.named_transformers_["num"]
ohe = ct.named_transformers_["cat"]

coefs = np.vstack([e.coef_[0] for e in clf.estimators_])
intercepts = np.array([e.intercept_[0] for e in clf.estimators_])

export = {
    "estrategia": str(mejor["estrategia"]),
    "C": float(mejor["C"]),
    "clases": [str(c) for c in clf.classes_],
    "cols_num": cols_num,
    "cols_cat": cols_cat,
    "scaler_mean": [float(v) for v in scaler.mean_],
    "scaler_scale": [float(v) for v in scaler.scale_],
    "cat_niveles": {
        col: {"base": str(cats[0]), "restantes": [str(c) for c in cats[1:]]}
        for col, cats in zip(cols_cat, ohe.categories_)
    },
    "coef": [[float(v) for v in fila] for fila in coefs],
    "intercept": [float(v) for v in intercepts],
    "sklearn_version": sklearn.__version__,
}
with open(os.path.join(CARPETA_RELEASE, "modelo_web.json"), "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False)

metricas = {
    "estrategia": str(mejor["estrategia"]),
    "C": float(mejor["C"]),
    "f1_macro_test": float(mejor["f1_macro_test"]),
    "f1_macro_train": float(mejor["f1_macro_train"]),
    "accuracy_test": float(mejor["accuracy_test"]),
    "accuracy_train": float(mejor["accuracy_train"]),
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "clases": [str(c) for c in clf.classes_],
    "sklearn_version": sklearn.__version__,
}
with open(os.path.join(CARPETA_RELEASE, "metricas.json"), "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

print("Contenido de", CARPETA_RELEASE)
for archivo in sorted(os.listdir(CARPETA_RELEASE)):
    tam = os.path.getsize(os.path.join(CARPETA_RELEASE, archivo))
    print(f"   {archivo:34s} {tam/1024:8.1f} KB")

In [ ]:
def vectorizar(fila):
    v = [(float(fila[c]) - export["scaler_mean"][i]) / export["scaler_scale"][i]
         for i, c in enumerate(cols_num)]
    for c in cols_cat:
        v += [1.0 if str(fila[c]) == n else 0.0 for n in export["cat_niveles"][c]["restantes"]]
    return np.array(v)

def predecir_desde_json(fila):
    z = coefs @ vectorizar(fila) + intercepts
    sig = 1.0 / (1.0 + np.exp(-z))
    return sig / sig.sum()

muestra = X_test.head(200)
desde_json = np.array([predecir_desde_json(r) for _, r in muestra.iterrows()])
desde_sklearn = modelo_final.predict_proba(muestra)

diferencia = np.abs(desde_json - desde_sklearn).max()
coincidencia = (desde_json.argmax(1) == desde_sklearn.argmax(1)).mean()

print(f"Diferencia maxima de probabilidad : {diferencia:.2e}")
print(f"Clases predichas coincidentes     : {coincidencia:.1%}")

assert diferencia < 1e-9, "El JSON exportado NO reproduce el modelo de scikit-learn"
print("\nVerificacion superada: el artefacto portable es equivalente al modelo entrenado.")

---

**Siguiente paso:** `05_discusion_y_conclusiones.ipynb`.